# Lab 08 — Inside an LLM, and RAG
**CE/ARCE 2XX · AI Literacy for Civil and Architectural Engineering** · Week 8 lab · about 75 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/labs/lab08-inside-llms-and-rag/Lab08_Inside_LLMs_and_RAG.ipynb)

Last week you saw how a large language model (LLM) works on slides. Today you open one up. You will see the **tokens** it reads, the **probabilities** it assigns to the next token, how **temperature** changes what it writes, and how sentences become **vectors** (embeddings). Then you will build a small **retrieval-augmented generation (RAG)** system that looks up OSHA excavation rules before answering, and you will check its answer against the real table.

### What you will be able to do
1. Split engineering text into tokens and explain why token counts matter (context window, cost, speed).
2. Read a model's top-5 next-token probabilities and explain how temperature changes generated text.
3. Turn sentences into embedding vectors and compare their meaning with cosine similarity.
4. Build a mini-RAG pipeline (chunk → retrieve → prompt → answer with citations) over OSHA Subpart P, and judge when retrieval helps and when it fails.

### Before you start
1. Click **Open in Colab** above (or open this file from the course GitHub page).
2. In Colab, choose **File → Save a copy in Drive**. Work in *your copy* so nothing is lost.
3. Run cells from top to bottom with **Shift + Enter**. Cells marked `# TODO` are yours to complete.

**Runtime:** the default CPU runtime is fine.
**Data:** `osha/osha_1926_subpartP_excavations.txt` — OSHA 29 CFR 1926 Subpart P (Excavations), U.S. Government work. A text copy of Table B-1 is included. For learning only; use the current official eCFR text for real work.

**AI-use label:** **AI-Assisted** — you may ask an AI tool for explanations or debugging help. Fill in the AI-use acknowledgment at the end. You are responsible for every line.

In [ ]:
# Setup: run this cell first (Shift + Enter). It loads libraries and a helper to fetch course data.
import os, urllib.request
import gc, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
BASE_URL = "https://raw.githubusercontent.com/TensorBuildLab/AI-Literacy-CE-ARCE/main/data/"

def get_data(name):
    """Download a course data file once and return its local path, e.g. get_data("concrete_strength.csv")."""
    if not os.path.exists(name):
        folder = os.path.dirname(name)
        if folder:
            os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + name, name)
    return name

print("Setup complete.")

**Start the download now (first 5 minutes of lab).** The next cell downloads **Qwen2.5-0.5B-Instruct**, the
small model from Lab 01 (about 1 GB the first time, 1–2 minutes). Read Part 1 while it runs. Model downloads happen
on Google's computers, not on your laptop or your phone's data plan.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)            # the tokenizer: text -> token ID numbers
model = AutoModelForCausalLM.from_pretrained(MODEL)   # the model: token IDs -> next-token scores (used in Part 2)
model.eval()     # "evaluation mode": we use the model, we do not train it
print("Loaded", MODEL)

## Part 1 — Tokens: how an LLM reads text (8 min)
An LLM never sees letters or words. A **tokenizer** first cuts text into **tokens** — whole words, pieces of
words, digits, or symbols — and turns each token into an ID number. The model only ever works with those numbers.

In [ ]:
print("Vocabulary size:", len(tok), "different tokens")

def show_tokens(text):
    """Print how the tokenizer splits `text`, and return the number of tokens."""
    ids = tok(text)["input_ids"]
    pieces = [tok.decode([i]) for i in ids]
    print(f"{len(ids):3d} tokens | {text}")
    print("            ", pieces)
    return len(ids)

show_tokens("concrete")
show_tokens(" concrete")          # the same word with a space in front
show_tokens("superplasticizer")
show_tokens("The water-cement ratio of this mix is 0.45.")

Why is "concrete" **2** tokens but " concrete" only **1**? This tokenizer glues the space to the **front** of
the next word (`' ratio'`, `' mix'`), so its vocabulary mostly stores words *with* their leading space. In the middle
of a sentence, ` concrete` is one common token. At the very start of a text there is no space, so the word is
split (`con` + `crete`). Rarer words like *superplasticizer* are cut into pieces either way, and numbers are split
digit by digit.

> 💡 **Hint:** Put each string in quotes, separated by commas. Try one with symbols or digits.

In [ ]:
# TODO: write three engineering strings of your own and count their tokens.
# Ideas: a unit ("kN·m", "kilonewton-meters"), a steel shape ("W14x90"), a long word ("geotechnical"),
# a code reference ("29 CFR 1926.652"), or a sentence from a spec.
my_texts = ["...", "...", "..."]
for t in my_texts:
    show_tokens(t)

### How long is a real document?
Models have a **context window** — the maximum number of tokens they can read at once — and paid AI services
charge **per token**. Let's count the tokens in the OSHA excavation rules we will use in Part 5.

In [ ]:
osha_text = open(get_data("osha/osha_1926_subpartP_excavations.txt"), encoding="utf-8").read()
n_words = len(osha_text.split())
n_tokens = len(tok(osha_text)["input_ids"])
print(f"OSHA Subpart P: {n_words:,} words -> {n_tokens:,} tokens  ({n_tokens / n_words:.2f} tokens per word)")

**✍️ Question:** Which of your strings needed the most tokens per word, and why do you think that is? Why would an engineer care how many tokens a 300-page specification has?

*Your answer:* 

## Part 2 — Next-token probabilities (10 min)
An LLM does one thing over and over: given the tokens so far, it gives a **score (logit)** to every token in its
vocabulary. The **softmax** function turns those scores into **probabilities** that add up to 1. Then one token is
picked, added to the text, and the process repeats.

The model you loaded at the start is the thing that computes those scores. How big is it?

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL}: {n_params / 1e6:.0f} million parameters")

Our test prompt is a beam formula you know: M = wL²/8, where **w is the uniform load (kN/m)** and L is the span (m). Here `w` is a load, not a model weight.

In [ ]:
def next_token_table(prompt, k=5):
    """Return a table with the k most likely next tokens after `prompt` and their probabilities."""
    ids = tok(prompt, return_tensors="pt")
    with torch.no_grad():                               # no training, just a forward pass
        logits = model(**ids).logits[0, -1].float()     # scores for the NEXT token (last position)
    probs = torch.softmax(logits, dim=-1)               # scores -> probabilities that sum to 1
    top = torch.topk(probs, k)
    return pd.DataFrame({"next token": [repr(tok.decode([int(i)])) for i in top.indices],
                         "probability": [round(float(p), 3) for p in top.values]})

beam_prompt = ("The maximum bending moment of a simply supported beam with a uniform load "
               "is w L squared divided by")
next_token_table(beam_prompt)

Surprise: the most likely next token is just a **space** `' '`. This tokenizer writes every digit as its own
token, and the space in front of a number is a separate token too. So the model is really saying "a number comes
next". Let's add the space to the prompt and look one step further: **which digit** does it expect?

In [ ]:
table = next_token_table(beam_prompt + " ")
display(table)

plt.figure(figsize=(6, 3))
plt.barh(table["next token"][::-1], table["probability"][::-1], color="#1E5288")
plt.xlabel("Probability (–)")
plt.ylabel("Next token")
plt.title("Which number comes after \"w L squared divided by\"?")
plt.tight_layout()
plt.show()

**✍️ Question:** Which digit has the highest probability? Is it the correct engineering answer (M = wL²/8)? What probability does the model give to the correct digit, and what does that tell you about using its most likely answer?

*Your answer:* 

> 💡 **Hint:** `display()` shows a table nicely. Example: "The SI unit of stress is the" — what should come next?

In [ ]:
# TODO: test the model's "knowledge" with two prompts of your own. End each prompt right before the word
# you want the model to predict (no period at the end).
prompt_a = "Concrete is strong in compression but weak in"
prompt_b = "..."
display(next_token_table(prompt_a))
display(next_token_table(prompt_b))

**✍️ Question:** A token can have a high probability and still be wrong. Explain in one or two sentences why "most likely next token" is not the same as "true".

*Your answer:* 

## Part 3 — Temperature: turning the randomness up and down (10 min)
When the model **samples** the next token, **temperature (T)** reshapes the probabilities first: the scores are
divided by T before the softmax.
- **Low T (e.g., 0.2):** the top token gets almost all the probability → predictable, repetitive text.
- **T = 1:** the model's own probabilities.
- **High T (e.g., 1.5):** probabilities get flatter → unlikely tokens get picked more often → creative or nonsense.

First, *see* the effect on the digit after "w L squared divided by" from Part 2.

In [ ]:
ids = tok(beam_prompt + " ", return_tensors="pt")
with torch.no_grad():
    logits = model(**ids).logits[0, -1].float()
top8 = torch.topk(logits, 8).indices
labels = [repr(tok.decode([int(i)])) for i in top8]

fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
for ax, T in zip(axes, [0.2, 1.0, 1.5]):
    p = torch.softmax(logits / T, dim=-1)[top8]
    ax.bar(range(8), p.numpy(), color="#AB0520")
    ax.set_xticks(range(8)); ax.set_xticklabels(labels, rotation=60, fontsize=8)
    ax.set_title(f"T = {T}")
    ax.set_xlabel("Candidate next token")
axes[0].set_ylabel("Probability (–)")
fig.suptitle("The digit after \"w L squared divided by\": same scores, different temperatures")
plt.tight_layout(); plt.show()

At low temperature the model picks its favorite digit most of the time, even when that digit is wrong. At high temperature the correct `8` gets picked more often, but so do all the other wrong digits.

Now let the model write. `sample_completions` writes a short continuation three times, each with a
different **random seed** (a starting number for the random generator). Same seed + same settings → same text, so
your results are reproducible.

In [ ]:
def sample_completions(prompt, temperature, seeds=(1, 2, 3), max_new_tokens=20, top_p=1.0):
    """Sample one continuation per seed at the given temperature; return a list of strings."""
    ids = tok(prompt, return_tensors="pt")
    outs = []
    for s in seeds:
        torch.manual_seed(s)
        with torch.no_grad():
            out = model.generate(**ids, do_sample=True, temperature=temperature, top_k=0, top_p=top_p,
                                 max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
        outs.append(tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    return outs

story_prompt = "The most common cause of cracks in a concrete slab is"
for T in [0.2, 1.0, 1.5]:
    print(f"--- T = {T}")
    for text in sample_completions(story_prompt, T):
        print("   ", text.replace("\n", " "))

Two things that look strange, but are normal for a small model:
- **Quiz-style text ("____. A. … B. …").** We gave the model a raw sentence, not a chat message. So it simply
  continues the text the way similar text continued in its training data, and much of that was exam questions.
- **Word salad at T = 1.5.** With no cutoff, any of the 151,665 tokens can be picked. When the probabilities are
  this flat, the **random seed** decides more than your prompt does, so the same seeds give similar junk for
  different prompts.

Many chat tools also use a **top-p cutoff**: they only pick from the most likely tokens that together make up a
share *p* of the probability, and drop the long tail of unlikely tokens. Same T = 1.5 and same seeds, but only the
tokens that make up the top 50% of the probability (`top_p=0.5`):

In [ ]:
for text in sample_completions(story_prompt, 1.5, top_p=0.5):
    print("   ", text.replace("\n", " "))

> 💡 **Hint:** Keep the prompt unfinished so the model continues it. Very high T (2.0) usually produces nonsense.

In [ ]:
# TODO: pick your own prompt and ONE temperature (try 0.7 or 2.0) and look at three samples.
my_prompt = "..."
my_T = 0.7
for text in sample_completions(my_prompt, my_T):
    print(text.replace("\n", " "))

**✍️ Question:** Which temperature would you use (a) for an AI tool that drafts a calculation report, and (b) for brainstorming names for a new campus building? Use what you saw above.

*Your answer:* 

## Part 4 — Embeddings: meaning as a vector (12 min)
An **embedding** is a list of numbers (a vector) that represents the meaning of a piece of text. Texts with
similar meaning get vectors that point in similar directions. We measure that with **cosine similarity**:
1 = same direction (very similar), 0 = unrelated.

Inside an LLM, **every token** gets its own vector (896 numbers in Qwen2.5-0.5B). A **sentence-embedding model**
averages its token vectors into **one vector per sentence or passage**, and that single vector is what search
systems compare. We use **all-MiniLM-L6-v2**, a small model (about 90 MB) made for this. It gives each token a
384-number vector, and we **average** them (mean pooling) to get one 384-number vector per sentence.

In [ ]:
from transformers import AutoModel

EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
emb_tok = AutoTokenizer.from_pretrained(EMB_MODEL)
emb_model = AutoModel.from_pretrained(EMB_MODEL)
emb_model.eval()

def embed(sentences):
    """Return one unit-length embedding vector (384 numbers) per sentence, as a NumPy array."""
    batch = emb_tok(sentences, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        token_vectors = emb_model(**batch).last_hidden_state          # (sentences, tokens, 384)
    mask = batch["attention_mask"].unsqueeze(-1).float()               # ignore padding tokens
    sentence_vectors = (token_vectors * mask).sum(1) / mask.sum(1)     # mean pooling
    sentence_vectors = torch.nn.functional.normalize(sentence_vectors, dim=1)   # length 1
    return sentence_vectors.numpy()

v = embed(["The concrete cylinder failed at 32 MPa."])
print("Shape:", v.shape, " First 5 numbers:", np.round(v[0, :5], 3))

For unit-length vectors, cosine similarity is just the **dot product**: multiply matching numbers and add them up. With a matrix of vectors, `V @ V.T` gives every pair at once.

In [ ]:
sentences = [
    "Adding more water to the mix lowers the compressive strength of concrete.",       # concrete
    "A higher water-cement ratio produces weaker concrete.",                           # concrete
    "Workers must not enter a trench deeper than 5 feet without cave-in protection.",  # trench safety
    "Excavations need shoring or sloping so the walls do not collapse on people.",     # trench safety
    "The chiller and air handlers use most of the building's electricity in summer.",  # HVAC energy
    "Cooling equipment dominates the energy bill of the office tower in July.",        # HVAC energy
]
V = embed(sentences)
sim = V @ V.T                                   # cosine similarity of every pair

short = ["water→weak", "w/c→weak", "trench 5 ft", "shoring", "chiller", "cooling bill"]
plt.figure(figsize=(5.5, 4.5))
plt.imshow(sim, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Cosine similarity (–)")
plt.xticks(range(6), short, rotation=45, ha="right"); plt.yticks(range(6), short)
for i in range(6):
    for j in range(6):
        plt.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", fontsize=8,
                 color="white" if sim[i, j] > 0.6 else "black")
plt.title("Sentence meaning similarity (MiniLM)")
plt.tight_layout(); plt.show()

**✍️ Question:** Look at sentence pairs 3–4 and 5–6: they share almost no words. Are their similarities high or low compared with pairs from different topics? What does that tell you about embeddings versus simple word matching?

*Your answer:* 

> 💡 **Hint:** Ask about one of the three topics *without* reusing its words, e.g. a question about air conditioning power.

In [ ]:
# TODO: write a "search question" and find which of the six sentences is closest in meaning.
query = "..."
scores = embed([query]) @ V.T          # similarity of the query to each sentence
best = int(np.argmax(scores))
print(np.round(scores, 2))
print("Closest sentence:", sentences[best])

## Part 5 — A mini-RAG over OSHA excavation rules (25 min)
**Retrieval-augmented generation (RAG)** = before the model answers, *your code* finds the most relevant passages
in a trusted document and pastes them into the prompt. The model is told to answer **only from those passages**
and to cite them. Four steps:

1. **Chunk** the document into small passages.
2. **Retrieve** the passages that best match the question.
3. **Prompt** the model with the question + retrieved passages.
4. **Generate** an answer with citations — and **check** it.

Our test question has a known answer. OSHA Subpart P, Appendix B, **Table B-1**: the maximum allowable slope for
**Type C** soil in excavations less than 20 ft deep is **1½H:1V (34°)** — 1.5 ft horizontal for every 1 ft
vertical.

### 5.1 Chunk the document
The file has one paragraph per line, and headings start with `##`. `make_chunks` keeps whole paragraphs together
until a chunk has at least 50 words, never mixes two sections, and keeps table rows (lines with `|`) with their
table title. Every chunk remembers its section, so answers can cite it.

In [ ]:
def make_chunks(text, min_words=50):
    """Split the OSHA text into chunks of whole paragraphs. Returns a list of {"section", "text"} dictionaries."""
    paragraphs, section = [], "Subpart P"
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith("## "):                       # a section heading
            section = line[3:]
        elif " | " in line and paragraphs and paragraphs[-1][0] == section:
            paragraphs[-1][1] += "\n" + line             # a table row: keep it with the line above
        else:
            paragraphs.append([section, line])
    chunks, current, current_section = [], [], None
    for section, para in paragraphs:
        if current and (section != current_section or len(" ".join(current).split()) >= min_words):
            chunks.append({"section": current_section, "text": "\n".join(current)})
            current = []
        current_section = section
        current.append(para)
    chunks.append({"section": current_section, "text": "\n".join(current)})
    return chunks

chunks = make_chunks(osha_text)
print(len(chunks), "chunks")
print("Example chunk #10 —", chunks[10]["section"])
print(chunks[10]["text"])

> 💡 **Hint:** `if "TABLE B-1" in c["text"]:` checks whether one string appears inside another. Indent the two print lines under it.

In [ ]:
# TODO: find the chunk that contains Table B-1. Loop over the chunks with their index (enumerate) and print the
# index and text of every chunk whose text contains "TABLE B-1".
for i, c in enumerate(chunks):
    pass   # replace this line with an if statement

### 5.2 Retrieve with TF-IDF
**TF-IDF** (term frequency–inverse document frequency) is classic keyword search: a chunk scores high when it
contains the query's words, especially words that are rare in the rest of the document. scikit-learn turns every
chunk into a TF-IDF vector; the score is the cosine similarity between the query vector and each chunk vector.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(stop_words="english")       # ignore filler words like "the", "is", "for"
chunk_vectors = vectorizer.fit_transform([c["text"] for c in chunks])

def search(query, k=3):
    """Return the k chunks most similar to the query, best first, each with its score and index."""
    scores = cosine_similarity(vectorizer.transform([query]), chunk_vectors)[0]
    best = np.argsort(scores)[::-1][:k]
    return [{"rank": r + 1, "score": round(float(scores[i]), 3), "chunk": int(i), **chunks[i]}
            for r, i in enumerate(best)]

def show_results(results, chars=160):
    for r in results:
        print(f"#{r['rank']}  score {r['score']}  chunk {r['chunk']}  [{r['section'][:45]}]")
        print("     ", r["text"][:chars].replace("\n", " / "), "...")

def has_table_b1(results):
    """True if the Type C row of Table B-1 is in the retrieved text."""
    return any("Type C | 1 1/2:1" in r["text"] for r in results)

question = "What is the maximum allowable slope for an excavation in Type C soil less than 20 ft deep?"
results = search(question)
show_results(results)
print("\nTable B-1 retrieved?", has_table_b1(results))

Look at what came back. Retrieval is the step people forget to check. Let's see the question the way
TF-IDF sees it — the list of words it actually searches for:

In [ ]:
analyzer = vectorizer.build_analyzer()          # the function TF-IDF uses to cut text into words
table_chunk = next(c for c in chunks if "TABLE B-1" in c["text"])
print("Your question becomes:", analyzer(question))
print("\nTable B-1 chunk becomes:", analyzer(table_chunk["text"]))

**✍️ Question:** Compare the two word lists. Name at least two reasons the keyword search missed (or under-ranked) Table B-1. Hint: what happened to the letter "C"? Is "slope" the same word as "slopes" to TF-IDF? What about "ft" and "feet"?

*Your answer:* 

> 💡 **Hint:** Use plural "slopes" and "excavations", write "feet" instead of "ft", and keep the key words "maximum allowable".

In [ ]:
# TODO: write a better SEARCH QUERY (not a new question) using the words the document itself uses
# (look at the second word list above). Then check that Table B-1 is in the top 3.
better_query = "..."
good_results = search(better_query)
show_results(good_results)
print("\nTable B-1 retrieved?", has_table_b1(good_results))

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert has_table_b1(good_results), 'Table B-1 is not in your top 3 yet — reuse more of the table words.'
print('Table B-1 found!')

### Optional (5 min): does meaning-based search do better?
Real RAG systems usually search with **embeddings** (Part 4) instead of keywords: "chunk → **embed** → retrieve".
Embeddings know that "ft" and "feet" mean the same thing. Let's embed all the chunks with MiniLM and search with
the **original** question (about 30–60 s on a CPU).

In [ ]:
chunk_embeddings = embed([c["text"] for c in chunks])        # one 384-number vector per chunk
q_embedding = embed([question])
emb_scores = (q_embedding @ chunk_embeddings.T)[0]               # cosine similarity to every chunk
emb_best = np.argsort(emb_scores)[::-1][:3]
emb_results = [{"rank": r + 1, "score": round(float(emb_scores[i]), 3), "chunk": int(i), **chunks[i]}
               for r, i in enumerate(emb_best)]
print("Embedding search, original question:")
show_results(emb_results)
print("\nTable B-1 retrieved?", has_table_b1(emb_results))

# Free memory: we are done with the embedding model
del emb_model, emb_tok
gc.collect()

**✍️ Question:** Compare the embedding search with the TF-IDF search for the same original question. What did embeddings fix, and what did they not fix?

*Your answer:* 

### 5.3 Ask the model — without and with retrieval
`ask()` is the same helper as in Lab 01 (greedy decoding, so answers are repeatable). We reuse the model that is
already loaded. `ask_with_sources()` builds a RAG prompt: numbered sources first, then strict instructions.

In [ ]:
from transformers import pipeline
chat = pipeline("text-generation", model=model, tokenizer=tok)

def ask(question, max_new_tokens=150, system="You are a helpful assistant for civil engineering students."):
    """Send one question to the small language model and return its answer as text."""
    messages = [{"role": "system", "content": system}, {"role": "user", "content": question}]
    out = chat(messages, max_new_tokens=max_new_tokens, do_sample=False)
    return out[0]["generated_text"][-1]["content"]

def build_rag_prompt(question, results):
    """Paste the retrieved chunks as numbered sources, then the question and the rules for answering."""
    sources = "\n\n".join(f"[{i + 1}] ({r['section']})\n{r['text']}" for i, r in enumerate(results))
    return (f"Sources:\n{sources}\n\n"
            f"Question: {question}\n"
            "Answer using ONLY the sources above. Give the value with units, and cite the source number in "
            "brackets, like [2]. If the sources do not contain the answer, say \"The sources do not say.\"")

def ask_with_sources(question, results):
    return ask(build_rag_prompt(question, results))

print(build_rag_prompt(question, good_results)[:700], "...")

Each of the next three cells takes about 20–60 seconds on a CPU. The correct answer is **1½H:1V (34°)**, OSHA Subpart P, Appendix B, Table B-1.

In [ ]:
answer_plain = ask(question)
print("=== 1) NO retrieval ===\n" + answer_plain)

In [ ]:
answer_bad = ask_with_sources(question, results)          # retrieval with the ORIGINAL question
print("=== 2) RAG with the ORIGINAL search — Table B-1 in the prompt?", has_table_b1(results), "===\n" + answer_bad)

In [ ]:
answer_good = ask_with_sources(question, good_results)     # retrieval with YOUR better query
print("=== 3) RAG with YOUR search — Table B-1 in the prompt?", has_table_b1(good_results), "===\n" + answer_good)

In [ ]:
def mentions_correct_slope(text):
    """A rough automatic check: does the answer mention 1 1/2:1 (or 1.5:1) or 34 degrees?"""
    t = text.replace("½", " 1/2").lower()
    return bool(re.search(r"1\s*1/2\s*(h)?\s*:\s*1|1\.5\s*(h)?\s*:\s*1|34\s*(°|degree)", t))

pd.DataFrame({
    "setup": ["1) no retrieval", "2) RAG, original search", "3) RAG, your search"],
    "Table B-1 in prompt?": ["no", has_table_b1(results), has_table_b1(good_results)],
    "mentions 1½:1 or 34°?": [mentions_correct_slope(a) for a in [answer_plain, answer_bad, answer_good]],
})

**✍️ Question:** Compare the three answers with Table B-1 (1½H:1V, 34°). Which ones are correct? Did the cited source number actually contain the value the model gave? What does case 2 teach you about RAG?

*Your answer:* 

> 💡 **Hint:** First look at the retrieved chunks: if the right rule is not there, the model cannot use it. Correct answers: egress needed at 4 ft (1.22 m) or deeper, max 25 ft (7.62 m) lateral travel [§ 1926.651(c)(2)]; spoil at least 2 ft (0.61 m) from the edge [§ 1926.651(j)(2)].

In [ ]:
# TODO: ask your own OSHA excavation question with retrieval. Write a good search query too.
# Ideas: "How deep must a trench be before a ladder is required?" (look for "means of egress");
#        "How far from the edge must spoil piles be kept?" (look for "2 feet").
my_question = "..."
my_query = "..."
my_results = search(my_query)
show_results(my_results)
print("\n", ask_with_sources(my_question, my_results))

**✍️ Question:** Check your own question's answer against the retrieved text. Was it correct, and was the citation right? Name one situation in a real project where you would NOT rely on this mini-RAG.

*Your answer:* 

In [ ]:
# Free memory at the end of the lab
del chat, model
gc.collect()

---
## 🎟️ Exit ticket
Three short questions you answer **alone, without AI (AI-Free)**, during the last 10 minutes of lab — even if you have not finished every part above. Use your own words, 1–3 sentences each. (If you must leave early, answer them on your own before you submit.)

**1.** In your own words, what does an LLM compute at every step when it writes text?

*Your answer:* 

**2.** Why can a RAG system still give a wrong answer, even when the right document is in its collection? Give one reason you saw today.

*Your answer:* 

**3.** What is the maximum allowable slope for an excavation less than 20 ft deep in Type C soil, and where in OSHA Subpart P is it stated?

*Your answer:* 

---
## 🤖 AI-use acknowledgment
Did you use an AI tool (ChatGPT, Gemini, Copilot, Claude, Colab's AI features, etc.) while working on this notebook? (The small model that this notebook's own code runs does not count.)
- If **no**, write: *"I did not use AI tools for this assignment."*
- If **yes**, write: *"I acknowledge the use of [tool and version] to [specific use]. Prompts I used included [paste or summarize]. I checked the output by [how I verified it]. I am responsible for this work."*

Same template as the course [AI-use guide](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/ai-use-guide.md).

*Your statement:* 

## 📤 How to submit
**Due:** Sunday 11:59 pm (Arizona time) of this week, on Brightspace. Late work: see the syllabus late-work table (two grace days per semester, then −10% per day, up to 3 days).

1. **Runtime → Restart session and run all** — make sure everything runs without errors (cells that load a model take a few extra minutes).
2. **File → Download → Download .ipynb**.
3. Upload the `.ipynb` file to the matching assignment on **Brightspace**.